# 05. 제외 시각과 판단 근거

[02 정비 기록](02_maintenance_parts.ipynb), [03 LOT 비교](03_lot_comparison.ipynb), [04 강종 비교](04_grade_comparison.ipynb)를 바탕으로 사용자 지정 4개 시간 구간을 제외하고 22개 계측을 비교함.

이 검토표의 정상 0, 이상 1은 미확정이라 `LABEL`은 `NA`로 남김.


In [ ]:
import os

import numpy as np
import pandas as pd
from IPython.display import display

if os.path.isfile(os.path.join('data', 'README.md')):
    ROOT = os.getcwd()
elif os.path.isfile(os.path.join('..', 'data', 'README.md')):
    ROOT = os.path.abspath('..')
else:
    ROOT = os.path.abspath(os.path.join('..', '..'))

DATA_DIR = os.path.join(ROOT, 'data')
RAW_DIR = DATA_DIR

# data 바로 아래에 원본이 없으면 data/raw에서 읽음.
if not os.path.isfile(os.path.join(RAW_DIR, 'T-CR1-CAL01_온도.csv')):
    RAW_DIR = os.path.join(DATA_DIR, 'raw')
FIG_DIR = os.path.join(ROOT, 'figures', 'namelesspark')

TEMP_CSV = os.path.join(RAW_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(RAW_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(RAW_DIR, 'G-01_정기수리캘린더.csv')
EQP_CSV = os.path.join(RAW_DIR, 'G-01_설비마스터.csv')
PART_CSV = os.path.join(RAW_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(RAW_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(RAW_DIR, 'G-02_부품교체이력.csv')
WELD_CSV = os.path.join(RAW_DIR, 'T-CR1-WLD01_용접.csv')  # 현재 노트북에서는 읽지 않음.
ENC = 'utf-8-sig'

PROCESSED = os.path.join(DATA_DIR, 'processed')
OUT = os.path.join(PROCESSED, 'review', 'v1')
GRADE_OUT = os.path.join(PROCESSED, 'analysis', 'grade', 'v1')
SIGNAL_OUT = os.path.join(PROCESSED, 'analysis', 'signals', 'v1')

for directory in [OUT, GRADE_OUT, SIGNAL_OUT]:
    os.makedirs(directory, exist_ok=True)

pd.set_option('display.max_columns', 40)

pd.set_option('display.max_colwidth', 100)

## 1. LOT, 강종과 정비 기록

CAL01의 BM, TBM, CBM을 `WO_NO`로 부품교체와 연결함. 감지, 착수, 종료는 정비 시각이며 실제 부품교체 시각은 제공되지 않음.


In [ ]:
temp_source = pd.read_csv(TEMP_CSV, encoding=ENC)
event_source = pd.read_csv(EVENT_CSV, encoding=ENC)
repair_source = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp_source = pd.read_csv(EQP_CSV, encoding=ENC)
part_source = pd.read_csv(PART_CSV, encoding=ENC)
maint_source = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg_source = pd.read_csv(PARTCHG_CSV, encoding=ENC)

# 원본 행과 값은 유지하고 날짜 컬럼의 형식만 맞춤.
temp_source['MEAS_DT'] = pd.to_datetime(temp_source['MEAS_DT'], errors='raise')
event_source['EVT_DT'] = pd.to_datetime(event_source['EVT_DT'], errors='raise')
repair_source['STRT_DT'] = pd.to_datetime(repair_source['STRT_DT'], errors='raise')
repair_source['END_DT'] = pd.to_datetime(repair_source['END_DT'], errors='raise')
eqp_source['INSTALL_DT'] = pd.to_datetime(eqp_source['INSTALL_DT'], errors='raise')
maint_source['DETC_DT'] = pd.to_datetime(maint_source['DETC_DT'], errors='raise')
maint_source['STRT_DT'] = pd.to_datetime(maint_source['STRT_DT'], errors='raise')
maint_source['END_DT'] = pd.to_datetime(maint_source['END_DT'], errors='raise')

sources = {
    'temp': temp_source,
    'event': event_source,
    'repair': repair_source,
    'eqp': eqp_source,
    'part': part_source,
    'maint': maint_source,
    'partchg': partchg_source,
}

temp = sources["temp"]


In [ ]:
# LOT 시작 시각과 정확히 같은 강종변경 기록만 연결함.

lot_context = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), ROWS=("MEAS_DT", "size")
)

changes = event_source.loc[
    event_source["PLANT_CD"].eq("CR1") & event_source["EVT_TYPE"].eq("강종변경"),
    ["EVT_DT", "BEF_VAL", "AFT_VAL"],
].sort_values("EVT_DT")

lot_context = lot_context.merge(
    changes[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
    how="left", validate="many_to_one",
).rename(columns={"AFT_VAL": "GRADE"}).drop(columns="EVT_DT")

lot_context["DURATION_H"] = (lot_context["END"] - lot_context["START"]).dt.total_seconds() / 3600
lot_context = lot_context[["LOT_NO", "GRADE", "START", "END", "ROWS", "DURATION_H"]]
lot_context = lot_context.sort_values("START").reset_index(drop=True)

display(lot_context)


### 작업번호별 부품 연결

교체 없는 정비도 남김. 작업시간과 정지 분량은 구분함.


In [ ]:
# 부품교체 상세를 WO별로 모은 후 정비이력에 left join함.
CAL_TAG = "P1-CR1-CAL01"

maintenance = maint_source.loc[maint_source["EQP_TAG"].eq(CAL_TAG)].copy()
parts = partchg_source.loc[partchg_source["WO_NO"].isin(maintenance["WO_NO"])].copy()

parts = parts.merge(part_source, on="PART_CD", how="left", validate="many_to_one", indicator=True)

parts = parts.drop(columns="_merge")

part_summary_rows = []

for wo_no, group in parts.groupby("WO_NO", sort=True):
    part_summary_rows.append({
        "WO_NO": wo_no,
        "PART_RECORD_COUNT": len(group),
        "PART_CODES": " | ".join(group["PART_CD"].astype(str)),
        "PART_NAMES": " | ".join(group["PART_NM"].astype(str)),
        "PART_QTY_TOTAL": group["QTY"].sum(),
        "PART_CHANGE_REASONS": " | ".join(group["CHG_RSN"].astype(str)),
    })

part_summary = pd.DataFrame(part_summary_rows, columns=[
    "WO_NO", "PART_RECORD_COUNT", "PART_CODES", "PART_NAMES", "PART_QTY_TOTAL", "PART_CHANGE_REASONS",
])

maintenance = maintenance.merge(part_summary, on="WO_NO", how="left", validate="one_to_one")

maintenance["HAS_PART_RECORD"] = maintenance["PART_RECORD_COUNT"].notna()

for column in ["PART_RECORD_COUNT", "PART_QTY_TOTAL"]:
    maintenance[column] = maintenance[column].fillna(0).astype(int)

for column in ["PART_CODES", "PART_NAMES", "PART_CHANGE_REASONS"]:
    maintenance[column] = maintenance[column].fillna("")

# 작업 시간과 기록된 정지 분량은 다르며 정지 시각을 역산하지 않음.
maintenance["WORK_MIN"] = (maintenance["END_DT"] - maintenance["STRT_DT"]).dt.total_seconds() / 60

maintenance["DOWN_RECORD_STATUS"] = "정지시간 미기록"
maintenance.loc[maintenance["DOWN_MIN"].eq(0), "DOWN_RECORD_STATUS"] = "해당 정비의 정지시간 0 기록"
maintenance.loc[maintenance["DOWN_MIN"].gt(0), "DOWN_RECORD_STATUS"] = "정지시간 기록 있음; 시점 미확정"

maintenance["DOWN_TO_WORK_RATIO"] = maintenance["DOWN_MIN"].div(maintenance["WORK_MIN"].where(maintenance["WORK_MIN"].gt(0)))
maintenance["DOWN_EXCEEDS_WORK"] = maintenance["DOWN_MIN"].astype("Float64").gt(maintenance["WORK_MIN"].astype("Float64"))
maintenance["STOP_INTERVAL_KNOWN"] = False

part_details = maint_source.loc[maint_source["EQP_TAG"].eq(CAL_TAG)].merge(
    parts, on="WO_NO", how="inner", validate="one_to_many"
)

maintenance = maintenance.sort_values(["DETC_DT", "WO_NO"]).reset_index(drop=True)
part_details = part_details.sort_values(["STRT_DT", "WO_NO", "PART_CD"]).reset_index(drop=True)

maintenance_counts = maintenance.groupby("MNT_TYPE").agg(
    정비건수=("WO_NO", "size"), 교체있음=("HAS_PART_RECORD", "sum"), 부품행수=("PART_RECORD_COUNT", "sum"),
)
maintenance_counts["교체없음"] = maintenance_counts["정비건수"] - maintenance_counts["교체있음"]

display(maintenance_counts.reindex(["BM", "TBM", "CBM"]))


## 2. 사용자 지정 시각 제외

| LOT | 제외할 시각 | 남길 경계 |
| --- | --- | --- |
| 240058 | 2024-06-09 08:00:00 이상 | 08:00 이전만 남김 |
| 240061 | 2024-06-15 02:30:00 미만 | 02:30부터 남김 |
| 240114 | 2024-11-04 20:00:00 이상 | 20:00 이전만 남김 |
| 240117 | 2024-11-13 09:00:00 미만 | 09:00부터 남김 |

240058은 정기수리 정지, 240061은 정기수리 후 재가동으로 판단함. 표는 실제 날짜와 시각이며, 원본을 바꾸지 않고 `EXCLUDED`로 표시함.

분 평균이나 빈 시각을 만들지 않고, 240038의 높은 출력도 추가 제외하지 않음.


In [ ]:
USER_EXCLUSIONS = [
    (240058, "2024-06-09 08:00:00", "at_or_after", "사용자 지정: 노 내림 구간 제외"),
    (240061, "2024-06-15 02:30:00", "before", "사용자 지정: 승온 구간 제외"),
    (240114, "2024-11-04 20:00:00", "at_or_after", "사용자 지정: 정기수리 전 하강 구간 제외"),
    (240117, "2024-11-13 09:00:00", "before", "사용자 지정: 정기수리 후 승온 구간 제외"),
]

row_review = temp.copy()
row_review["EXCLUDED"] = False
row_review["EXCLUSION_REASON"] = ""

exclusion_rows = []

for lot, boundary, direction, reason in USER_EXCLUSIONS:
    boundary = pd.Timestamp(boundary)

    if direction == "at_or_after":
        time_mask = row_review["MEAS_DT"].ge(boundary)
    else:
        time_mask = row_review["MEAS_DT"].lt(boundary)

    hit = row_review["LOT_NO"].eq(lot) & time_mask
    row_review.loc[hit, "EXCLUDED"] = True
    row_review.loc[hit, "EXCLUSION_REASON"] = reason
    exclusion_rows.append({
        "LOT_NO": lot, "boundary": boundary, "direction": direction, "excluded_rows": int(hit.sum()),
    })

exclusion_summary = pd.DataFrame(exclusion_rows)

row_review["GRADE"] = row_review["LOT_NO"].map(lot_context.set_index("LOT_NO")["GRADE"])

row_counts = pd.DataFrame({
    "구분": ["원본", "사용자 지정 제외", "제외 후 남는 행"],
    "행수": [len(row_review), int(row_review["EXCLUDED"].sum()),
             int((~row_review["EXCLUDED"]).sum())],
})

display(exclusion_summary)
display(row_counts)

## 3. 강종별 22개 계측 비교

중앙값, 5~95백분위, 최솟값, 최댓값은 관측 요약이며 정상 허용치가 아님. CP-Z4M은 정의와 단위가 미확인인 보조계측임.

[온도 데이터사전](https://app.notion.com/p/b5de609a45d083f89c01014faf0e4dda)은 강종군과 이동 폭을 재구성 가정으로 설명하며, CLN-IN과 건조 Z1, Z2는 강종에 반응하지 않는다고 명시함.

SGCC, DP590, SPCD는 각각 1개 LOT라 강종 효과와 LOT 상태를 분리하기 어려움. SPHC 240091의 TC-Z7, TC-Z8 수준 차이는 [04의 비교](04_grade_comparison.ipynb)를 근거로 검토함.


In [ ]:
SENSOR_COLUMNS = []

for column in temp.columns:
    if column not in ['LOT_NO', 'MEAS_DT']:
        SENSOR_COLUMNS.append(column)

retained = row_review.loc[~row_review["EXCLUDED"]].copy()

GRADE_INDEPENDENT_IN_DICTIONARY = {"CLN-IN", "TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2"}
GRADE_RELEVANCE_SOURCE = "https://app.notion.com/p/b5de609a45d083f89c01014faf0e4dda"

sensor_context = {}

for sensor in SENSOR_COLUMNS:
    if sensor in GRADE_INDEPENDENT_IN_DICTIONARY:
        relevance = "사전상 강종 반응 없음; 관측 확인"
    elif sensor == "CP-Z4M":
        relevance = "강종 영향 검토; 보조계측 정의 확인 우선"
    else:
        relevance = "강종 영향 검토; 사전의 수준 이동은 재구성 가정"

    if sensor == "CP-Z4M":
        role, paired, needed = "보조계측", "", "정의·단위·생성 규칙, 다른 계측과의 관계"
    elif sensor == "CP-Z4":
        role, paired, needed = "분위기 계측", "", "계측 정의·단위, 강종별 목표 CP·허용폭·지속시간, 가스 운전·퍼지 기록"
    elif sensor == "CLN-IN":
        role, paired, needed = "세정 온도", "", "강종별 세정 목표온도·허용폭, 약품·속도·초기 운전상태"
    elif sensor in ["TC-Z9", "TC-OUT1", "TC-OUT2"]:
        role, paired, needed = "주기 변화가 관측된 온도", "", "강종별 목표·허용폭, 정상 주기·위상·진폭, 속도·유량·운전단계"
    elif sensor.startswith("OP-"):
        role, paired, needed = "제어출력", sensor.replace("OP-", "TC-"), "같은 존 TC 설정값, 자동·수동 모드, 출력 지속 허용시간, 실제 히터·밸브 동작"
    else:
        role, paired, needed = "존 온도", sensor.replace("TC-", "OP-"), "강종·존별 설정온도, 허용편차·지속시간, 제어모드·라인속도·운전단계"

    sensor_context[sensor] = {"role": role, "paired": paired, "needed": needed, "relevance": relevance}



In [ ]:
lot_sensor_rows = []

for lot, group in retained.groupby("LOT_NO", sort=True):
    for sensor in SENSOR_COLUMNS:
        role = sensor_context[sensor]["role"]
        paired = sensor_context[sensor]["paired"]
        needed = sensor_context[sensor]["needed"]

        summary_row = {
            "LOT_NO": lot, "GRADE": group["GRADE"].iloc[0], "SENSOR": sensor,
            "RETAINED_ROWS": len(group),
        }

        valid = group[sensor].dropna()

        summary_row.update({
            "VALID_ROWS": int(group[sensor].notna().sum()),
            "MISSING_ROWS": int(group[sensor].isna().sum()),
            "MEDIAN": valid.median(), "Q05": valid.quantile(0.05), "Q95": valid.quantile(0.95),
            "MIN": valid.min(), "MAX": valid.max(),
        })

        summary_row.update({
            "SENSOR_ROLE": role, "PAIRED_SENSOR": paired,
            "GRADE_RELEVANCE": sensor_context[sensor]["relevance"], "GRADE_RELEVANCE_SOURCE": GRADE_RELEVANCE_SOURCE,
            "SUMMARY_ROLE": "관측 요약; 정상 허용치 아님",
        })

        lot_sensor_rows.append(summary_row)

lot_sensor_summary = pd.DataFrame(lot_sensor_rows)

# 지속 수준 차이의 통계, 구간 비교는 04에서 관리하고 여기서는 근거만 참조함.
sphc_level_notes = {}

for sensor in ["TC-Z7", "TC-Z8"]:
    sphc_level_notes[sensor] = "04_grade_comparison.ipynb의 SPHC 240091 지속 수준 차이 검토 참조; 원인·정상 허용치 미확정"

level_review_notes = []

for grade, sensor in zip(lot_sensor_summary["GRADE"], lot_sensor_summary["SENSOR"]):
    if grade == "SPHC":
        level_review_notes.append(sphc_level_notes.get(sensor, ""))
    else:
        level_review_notes.append("")

lot_sensor_summary["SUSTAINED_LEVEL_REVIEW"] = level_review_notes



In [ ]:
phase_notes = {
    "SGCC": "240001 단일 LOT. 반복 파형의 독립 정상 기준 미확정",
    "DP590": "240058 단일 LOT. 08:00 이전에도 정기수리 준비 과정이 포함됐는지 확인 필요",
    "SPCD": "240117 단일 LOT. 09:00 이후에도 정비 작업 겹침이 남아 있음",
    "SPHC": "240044·061·069·091 비교. 061 재가동을 구분하고 091 TC-Z7/Z8의 지속 수준 차이도 확인해야 함",
    "SPCE": "240037·121·133 비교. 반복 주기와 정비 감지 맥락을 구분해야 함",
    "SPCEN": "240038·114 비교. 114는 20:00 이후 제외했고 그 이전 종료 준비 여부도 확인 필요",
    "SPFH590": "240024·108 비교. 목표·속도 조건이 같은지는 미확정",
}

grade_sensor_rows = []

for grade, group in retained.groupby("GRADE", sort=True):
    lot_numbers = sorted(group["LOT_NO"].unique())

    lot_number_text = []

    for lot_number in lot_numbers:
        lot_number_text.append(str(lot_number))

    for sensor in SENSOR_COLUMNS:
        per_lot = lot_sensor_summary.loc[
            lot_sensor_summary["GRADE"].eq(grade) & lot_sensor_summary["SENSOR"].eq(sensor)
        ]

        medians = per_lot["MEDIAN"].dropna()

        role = sensor_context[sensor]["role"]
        paired = sensor_context[sensor]["paired"]
        needed = sensor_context[sensor]["needed"]

        can_compare = len(lot_numbers) > 1 and len(medians) > 1

        if can_compare:
            comparison_scope = "동일 강종 LOT 간 비교 가능; 정상 기준 미확정"
        else:
            comparison_scope = "동일 강종 LOT 간 비교 근거 부족; LOT 내부 관찰만 가능"

        if grade == "SPHC":
            sustained_note = sphc_level_notes.get(sensor, "")
        else:
            sustained_note = ""

        summary_row = {
            "GRADE": grade, "SENSOR": sensor, "SENSOR_ROLE": role,
            "GRADE_RELEVANCE": sensor_context[sensor]["relevance"], "GRADE_RELEVANCE_SOURCE": GRADE_RELEVANCE_SOURCE,
            "LOT_COUNT": len(lot_numbers), "VALID_LOT_COUNT": len(medians),
            "LOT_LIST": " | ".join(lot_number_text),
            "RETAINED_ROWS": len(group),
        }

        valid = group[sensor].dropna()

        summary_row.update({
            "VALID_ROWS": int(group[sensor].notna().sum()),
            "MISSING_ROWS": int(group[sensor].isna().sum()),
            "MEDIAN": valid.median(), "Q05": valid.quantile(0.05), "Q95": valid.quantile(0.95),
            "MIN": valid.min(), "MAX": valid.max(),
        })

        summary_row.update({
            "LOT_MEDIAN_MIN": medians.min(), "LOT_MEDIAN_MAX": medians.max(),
            "LOT_MEDIAN_RANGE": medians.max() - medians.min(),
            "LOT_MEDIAN_STD": medians.std(ddof=1),
            "CROSS_LOT_COMPARISON_AVAILABLE": can_compare,
            "COMPARISON_SCOPE": comparison_scope,
            "PAIRED_SENSOR": paired, "PHASE_CAUTION": phase_notes.get(grade, "운전단계 확인 필요"),
            "SUSTAINED_LEVEL_REVIEW": sustained_note,
            "REQUIRED_DOMAIN_INPUT": needed,
            "NORMAL_RULE_STATUS": "미확정", "ANOMALY_RULE_STATUS": "미확정",
            "REVIEW_STATUS": "미확정", "MODEL_INPUT_ALLOWED": False,
            "SUMMARY_ROLE": "관측 요약; 정상 허용치 아님",
            "EVIDENCE_SOURCE": "원본 온도·강종변경 + 사용자 지정 시간 제외; docs/02-data-contract.md",
        })

        grade_sensor_rows.append(summary_row)

grade_sensor_review = pd.DataFrame(grade_sensor_rows)

display(grade_sensor_review[[
    "GRADE", "SENSOR", "LOT_COUNT", "VALID_ROWS", "MISSING_ROWS", "MEDIAN", "Q05", "Q95",
    "LOT_MEDIAN_RANGE", "GRADE_RELEVANCE", "COMPARISON_SCOPE", "REQUIRED_DOMAIN_INPUT",
]])
display(lot_sensor_summary)

## 4. 10초 이상 이어진 변화 찾기

과거 5분 중앙값과 LOT별 센서 변동 규모를 기준으로, 실제 1초 간격으로 10초 이상 이어진 변화를 찾음. 탐색 기준은 공정 허용치가 아니며, 제어반응과 주기 변화 등을 구분해 검토함.


### 변화 탐색 기준값


In [ ]:
import json

TEMPORAL_REVIEW_RULES = {
    "version": "1.0",
    "reference": "이전 300초 유효 관측의 중앙값, 현재 시점 제외, 최소 60관측",
    "sustained_threshold": "max(6*잔차MAD시그마, 12*실제1초차분MAD시그마, 0.10*(LOT Q95-Q05), 부동소수점바닥)",
    "MAD_sigma": "1.4826 * median(abs(x-median(x)))",
    "duration": "같은 잔차 방향, 실제 1초 연속, END-START >= 10초 (최소 11관측)",
    "instant_summary_threshold": "max(12*실제1초차분MAD시그마, 3*Q99(abs(실제1초차분)), 0.25*(LOT Q95-Q05), 부동소수점바닥)",
    "periodicity": "연속 유효 1분 평균 구간의 선형 추세 제거 후 FFT; 최소 3주기, 주기>=3분, 최대주파수전력비>=0.25, 지연상관>=0.60",
    "limits": [
        "모든 문턱은 탐색 설정이며 공정 허용치나 정상·이상 정답이 아님",
        "전체 LOT로 구한 MAD·분위수·주기성은 사후 검토 통계이며 모델 입력에 그대로 사용하면 안 됨",
        "센서값·관측 공백은 보간하지 않으며 시간 공백이 지속 구간을 끊음",
        "TC 20도 왕복, OP 0·100, CP 공통 밴드를 단독 고장 규칙으로 사용하지 않음",
        "LOT 시작부터 유지된 수준 차이는 이 탐색으로 놓칠 수 있어 같은 강종의 다른 LOT와 별도 비교 필요",
        "주기·제어반응 표시는 정상 확정이 아니라 검토 맥락임",
    ],
}

# 현재 관측을 제외한 과거 300초 중앙값을 기준으로 잔차를 계산함.
temporal_work = []

for lot, group in retained.groupby("LOT_NO", sort=True):
    frame = group.sort_values("MEAS_DT").reset_index(drop=True)

    matching = lot_context.loc[lot_context.LOT_NO.eq(lot)]

    grade = matching.iloc[0].GRADE

    seconds = frame.MEAS_DT.to_numpy(dtype="datetime64[s]").astype("int64")
    exact = frame.MEAS_DT.diff().dt.total_seconds().eq(1)

    for sensor in SENSOR_COLUMNS:
        values = frame[sensor]
        series = pd.Series(values.to_numpy(), index=pd.DatetimeIndex(frame.MEAS_DT))

        difference = values.diff().where(exact)
        # MAD는 결측과 무한값을 제외한 뒤 기존 NumPy 식으로 계산함.
        array = np.asarray(difference, dtype=float)
        array = array[np.isfinite(array)]
        noise = 0.0

        if len(array):
            noise = float(1.4826 * np.median(np.abs(array - np.median(array))))

        q05, q95 = values.quantile([0.05, 0.95])
        width = float(q95 - q05)
        floor = np.finfo(float).eps * max(float(values.abs().max()), 1e-15) * 32

        baseline = series.rolling("300s", closed="left", min_periods=60).median().reset_index(drop=True)
        residual = values - baseline

        array = np.asarray(residual, dtype=float)
        array = array[np.isfinite(array)]
        residual_sigma = 0.0

        if len(array):
            residual_sigma = float(1.4826 * np.median(np.abs(array - np.median(array))))

        threshold = max(6 * residual_sigma, 12 * noise, 0.10 * width, floor)
        instant_threshold = max(12 * noise, 3 * float(difference.abs().quantile(0.99)), 0.25 * width, floor)

        temporal_work.append({
            "lot": lot, "grade": grade, "sensor": sensor, "frame": frame, "seconds": seconds,
            "values": values, "series": series, "difference": difference, "noise": noise,
            "q05": q05, "q95": q95, "width": width, "baseline": baseline, "residual": residual,
            "residual_sigma": residual_sigma, "threshold": threshold, "instant_threshold": instant_threshold,
        })


### 가장 긴 유효 구간의 반복 주기


In [ ]:
# 가장 긴 연속 1분 평균 구간에서 추세를 빼고 FFT와 지연상관을 확인함.
# FFT는 주기 확인용이며 모델 학습이나 고장 판정이 아님.
for work in temporal_work:
    series = work["series"]

    minute = series.resample("1min").mean()
    valid = np.flatnonzero(minute.notna().to_numpy())
    blocks = []

    if len(valid):
        blocks = np.split(valid, np.flatnonzero(np.diff(valid) != 1) + 1)

    period = {"PERIODIC_FLAG": False, "PERIOD_MIN": None, "POWER_SHARE": None,
              "LAG_CORR": None, "SEGMENT_MINUTES": 0}
    work["period"] = period

    if not blocks:
        continue

    values = minute.iloc[max(blocks, key=len)].to_numpy()
    period["SEGMENT_MINUTES"] = len(values)

    if len(values) < 30:
        continue

    index = np.arange(len(values))
    values = values - np.polyval(np.polyfit(index, values, 1), index)

    if np.std(values) <= np.finfo(float).eps * max(1.0, abs(series.median())):
        continue

    frequencies = np.fft.rfftfreq(len(values), d=1)
    power = np.abs(np.fft.rfft(values)) ** 2
    eligible = (frequencies >= 3 / len(values)) & (frequencies <= 1 / 3)

    if not eligible.any():
        continue

    peak = np.flatnonzero(eligible)[np.argmax(power[eligible])]
    period_minutes = 1 / frequencies[peak]
    lag = int(round(period_minutes))

    share = 0.0

    if power[1:].sum():
        share = float(power[peak] / power[1:].sum())

    correlation = np.nan

    if lag < len(values) - 5:
        correlation = float(np.corrcoef(values[lag:], values[:-lag])[0, 1])

    period.update(PERIOD_MIN=float(period_minutes), POWER_SHARE=share, LAG_CORR=correlation,
                  PERIODIC_FLAG=bool(share >= 0.25 and correlation >= 0.60))


### 연속 관측 구간

실제 시각이 1초 간격이 아니면 구간을 나눔.


In [ ]:
# 유효 관측, 0, 출력 100 이상 상태가 실제 1초 간격으로 얼마나 이어지는지 기록함.
coverage_rows = []

for work in temporal_work:
    frame = work["frame"]
    values = work["values"]
    seconds = work["seconds"]
    sensor = work["sensor"]
    difference = work["difference"]
    baseline = work["baseline"]

    longest = {}
    masks = {"valid": values.notna(), "zero": values.eq(0)}

    if sensor.startswith("OP-"):
        masks["ge100"] = values.ge(100)

    for name, mask in masks.items():
        positions = np.flatnonzero(np.asarray(mask, dtype=bool))
        runs = []

        if len(positions):
            splits = np.flatnonzero((np.diff(positions) != 1) | (np.diff(seconds[positions]) != 1)) + 1
            runs = np.split(positions, splits)

        longest[name] = 0

        for run in runs:
            elapsed = int(seconds[run[-1]] - seconds[run[0]])
            longest[name] = max(longest[name], elapsed)

    ge100 = np.nan

    if sensor.startswith("OP-"):
        ge100 = longest["ge100"]

    cov = {
        "LOT_NO": int(work["lot"]), "GRADE": work["grade"], "SENSOR": sensor,
        "RETAINED_ROWS": len(frame), "VALID_ROWS": int(values.notna().sum()),
        "MISSING_ROWS": int(values.isna().sum()), "START": frame.MEAS_DT.min(), "END": frame.MEAS_DT.max(),
        "MIN": float(values.min()), "Q05": float(work["q05"]), "MEDIAN": float(values.median()),
        "Q95": float(work["q95"]), "MAX": float(values.max()), "MAX_EXACT_1S_ABS_CHANGE": float(difference.abs().max()),
        "DIFF_ROBUST_SIGMA": work["noise"], "RESIDUAL_ROBUST_SIGMA": work["residual_sigma"],
        "INSTANT_THRESHOLD": work["instant_threshold"], "SUSTAINED_THRESHOLD": work["threshold"],
        "INSTANT_THRESHOLD_EXCEEDING_POINTS": int(difference.abs().gt(work["instant_threshold"]).sum()),
        "BASELINE_VALID_ROWS": int(baseline.notna().sum()),
        "MAX_CONTIGUOUS_VALID_S": longest["valid"], "MAX_CONTIGUOUS_ZERO_S": longest["zero"],
        "MAX_CONTIGUOUS_GE100_S": ge100,
    }

    cov.update(work["period"])

    coverage_rows.append(cov)


### 10초 이상 이어진 이상 후보

과거 기준보다 높거나 낮은 방향이 유지된 구간을 남김.


In [ ]:
# 같은 방향의 잔차가 실제 1초 간격으로 10초 이상 이어진 구간만 남김.
candidate_rows = []

for work in temporal_work:
    frame = work["frame"]
    values = work["values"]
    seconds = work["seconds"]
    sensor = work["sensor"]
    residual = work["residual"]
    baseline = work["baseline"]
    threshold = work["threshold"]
    period = work["period"]

    masks = [("local_deviation_high_10s", residual.gt(threshold)),
             ("local_deviation_low_10s", residual.lt(-threshold))]

    for kind, mask in masks:
        positions = np.flatnonzero(np.asarray(mask.fillna(False), dtype=bool))
        runs = []

        if len(positions):
            splits = np.flatnonzero((np.diff(positions) != 1) | (np.diff(seconds[positions]) != 1)) + 1
            runs = np.split(positions, splits)

        for indices in runs:
            elapsed = int(seconds[indices[-1]] - seconds[indices[0]])

            if elapsed < 10:
                continue

            run_values = values.iloc[indices]
            run_times = frame.MEAS_DT.iloc[indices]
            run_residual = residual.iloc[indices]
            start, end = run_times.iloc[0], run_times.iloc[-1]

            boundary = ((start - frame.MEAS_DT.iloc[0]).total_seconds() <= 1800
                        or (frame.MEAS_DT.iloc[-1] - end).total_seconds() <= 1800)

            # 기존 분류 우선순서를 유지함. 제어반응이나 주기성은 정상 확정이 아님.
            if sensor == "CP-Z4M":
                category = "보조계측·생성관계 확인"
                reason = "TC-Z5와의 중복성/생성 관계를 먼저 확인해야 함"
            elif sensor.startswith("OP-"):
                category = "제어반응·운전조건 확인"
                reason = "출력의 급변·0·포화는 단독 고장 근거가 아니며 같은 존 TC·제어모드와 같이 검토해야 함"
            elif period["PERIODIC_FLAG"] or sensor in ["TC-Z9", "TC-OUT1", "TC-OUT2"]:
                category = "반복주기·운전패턴 확인"
                reason = "반복 주기 또는 기존 주기채널 맥락이 있어 진폭·위상·운전조건을 함께 검토해야 함"
            elif sensor == "TC-Z3":
                category = "제어반응·운전조건 확인"
                reason = "TC-Z3와 OP-Z3의 제어반응 맥락이며 20도 왕복만으로 이상을 정하지 않음"
            elif boundary:
                category = "LOT 경계 운전조건 확인"
                reason = "관측 시작/끝 30분 이내로 운전단계·강종 전환·종료 준비 가능성을 함께 검토해야 함"
            else:
                category = "이상 검토 우선"
                reason = "해당 LOT·센서 자체의 변동 규모에 비해 큰 국소 변화임. 원인·정상 허용치는 미확정"

            stats = {
                "start_value": float(run_values.iloc[0]), "end_value": float(run_values.iloc[-1]),
                "min": float(run_values.min()), "median": float(run_values.median()), "max": float(run_values.max()),
                "peak_to_peak": float(run_values.max() - run_values.min()), "observations": len(indices),
                "threshold": threshold, "past5m_median_reference_median": float(baseline.iloc[indices].median()),
                "median_residual": float(run_residual.median()), "max_abs_residual": float(run_residual.abs().max()),
                "exact_1s_difference_robust_sigma": work["noise"], "LOT_Q95_Q05_width": work["width"],
                "period_min": period["PERIOD_MIN"], "period_power_share": period["POWER_SHARE"],
                "period_lag_corr": period["LAG_CORR"],
            }

            candidate_rows.append({
                "LOT_NO": int(work["lot"]), "GRADE": work["grade"], "SENSOR": sensor, "START": start, "END": end,
                "KIND": kind, "AMPLITUDE": float(run_residual.abs().max()), "DURATION_S": elapsed,
                "OBSERVED_ROWS": len(indices), "MIN_VALUE": float(run_values.min()), "MAX_VALUE": float(run_values.max()),
                "MEDIAN_VALUE": float(run_values.median()), "BASELINE_MEDIAN": float(baseline.iloc[indices].median()),
                "THRESHOLD": threshold, "REVIEW_CATEGORY": category, "PERIODIC_CONTEXT": period["PERIODIC_FLAG"],
                "LOT_BOUNDARY_CONTEXT": boundary, "FAULT_LABEL": pd.NA,
                "EVIDENCE_REASON": reason + "; 통계적 탐색 후보이며 확정 이상 1이 아님",
                "STATS": json.dumps(stats, ensure_ascii=False),
            })


### 같은 강종의 다른 LOT와 비교


In [ ]:
temporal_coverage = pd.DataFrame(coverage_rows)

candidate_columns = [
    "LOT_NO", "GRADE", "SENSOR", "START", "END", "KIND", "AMPLITUDE", "DURATION_S",
    "OBSERVED_ROWS", "MIN_VALUE", "MAX_VALUE", "MEDIAN_VALUE", "BASELINE_MEDIAN", "THRESHOLD",
    "REVIEW_CATEGORY", "PERIODIC_CONTEXT", "LOT_BOUNDARY_CONTEXT", "FAULT_LABEL", "EVIDENCE_REASON", "STATS",
]

sustained_candidates = pd.DataFrame(candidate_rows, columns=candidate_columns).sort_values(
    ["LOT_NO", "START", "SENSOR", "KIND"]).reset_index(drop=True)

candidate_ids = []

for number in range(1, len(sustained_candidates) + 1):
    candidate_ids.append("T" + str(number).zfill(5))

sustained_candidates.insert(0, "CANDIDATE_ID", candidate_ids)

counts = sustained_candidates.groupby(["LOT_NO", "SENSOR"]).size()

candidate_counts = []

for row in temporal_coverage.itertuples():
    candidate_counts.append(int(counts.get((row.LOT_NO, row.SENSOR), 0)))

temporal_coverage["CANDIDATE_COUNT"] = candidate_counts

for group_key, indices in temporal_coverage.groupby(["GRADE", "SENSOR"]).groups.items():
    for index in indices:
        other_indices = []

        for other in indices:
            if other != index:
                other_indices.append(other)

        others = temporal_coverage.loc[other_indices, "MEDIAN"]
        temporal_coverage.loc[index, "SAME_GRADE_OTHER_LOTS"] = len(others)
        temporal_coverage.loc[index, "OTHER_LOT_MEDIANS_MEDIAN"] = others.median()
        temporal_coverage.loc[index, "MEDIAN_MINUS_OTHER_LOTS"] = temporal_coverage.loc[index, "MEDIAN"] - others.median()

display(sustained_candidates.groupby(['LOT_NO', 'SENSOR']).size().rename('지속후보수').reset_index())

# 기존 검토표와의 호환용 빈 컬럼. 과거 CP 공통범위 규칙은 실행하지 않음.
CALCULATE_CP_CANDIDATES = False
row_review['CP_CANDIDATE'] = pd.Series(pd.NA, index=row_review.index, dtype='boolean')
row_review['CP_RUN_ID'] = pd.Series(pd.NA, index=row_review.index, dtype='string')
cp_runs = pd.DataFrame(columns=['CP_RUN_ID', 'LOT_NO', 'START', 'END'])


## 5. 구간 앞뒤의 정비와 조업 기록

정비 감지와 작업, 부품교체, 정기수리, 조업이벤트를 직전, 구간 중, 직후로 연결함. 가까운 기록도 원인으로 자동 판정하지 않음.

`DOWN_MIN`은 정지 분량이며 실제 정지 시점은 없음. 정기수리는 계획이므로 전체 기간을 연속 정지로 보지 않음.


### 이상 후보와 LOT 전체 구간


In [ ]:
# 지속 후보와 모든 LOT의 운전 맥락을 구분함. LOT 범위 자체는 이상 구간이 아님.
lot_windows = retained.groupby('LOT_NO', as_index=False).agg(START=('MEAS_DT', 'min'), END=('MEAS_DT', 'max'))
lot_windows['GRADE'] = lot_windows.LOT_NO.map(lot_context.set_index('LOT_NO').GRADE)
lot_windows['SENSOR'] = 'ALL_SENSORS_CONTEXT'
lot_windows['KIND'] = 'LOT_CONTEXT_ONLY'

domain_intervals = pd.concat([sustained_candidates, lot_windows], ignore_index=True).sort_values(
    ['LOT_NO', 'START', 'END', 'SENSOR']).reset_index(drop=True)

interval_ids = []

for number in range(1, len(domain_intervals) + 1):
    interval_ids.append('DOM-' + str(number).zfill(4))

domain_intervals['INTERVAL_ID'] = interval_ids

# 근거 연결에 쓸 구간과 원본 표를 준비함.

context_candidates = domain_intervals.copy().reset_index(drop=True)

for column in ["START", "END"]:
    context_candidates[column] = pd.to_datetime(context_candidates[column], errors="raise")

if "INTERVAL_ID" not in context_candidates:
    context_ids = []

    for number in range(1, len(context_candidates) + 1):
        context_ids.append("CTX-" + str(number).zfill(4))

    context_candidates["INTERVAL_ID"] = context_ids

context_lots = lot_context.set_index("LOT_NO", verify_integrity=True)

context_events = event_source.loc[event_source.PLANT_CD.eq("CR1")].copy()
context_repair = repair_source.loc[repair_source.PLANT_CD.eq("CR1")].copy()
context_equipment = eqp_source.loc[eqp_source.EQP_TAG.eq(CAL_TAG)].copy()
context_maint = maintenance.loc[maintenance.EQP_TAG.eq(CAL_TAG)].copy()



### 정비 감지, 작업과 조업 원본 기록


In [ ]:
# 각 구간에서 설비, 정비 유형별 감지와 작업, 수리계획, 조업이벤트 순으로 확인함.
context_requests = []

for interval in context_candidates.itertuples(index=False):

    lot = context_lots.loc[interval.LOT_NO]

    base = {}

    for field in ["INTERVAL_ID", "LOT_NO", "GRADE", "SENSOR", "START", "END"]:
        base[field] = getattr(interval, field)

    for field in ["CANDIDATE_ID", "KIND", "AMPLITUDE", "DURATION_S", "THRESHOLD",
                  "MIN_VALUE", "MAX_VALUE", "BASELINE_MEDIAN", "REVIEW_CATEGORY",
                  "PERIODIC_CONTEXT", "LOT_BOUNDARY_CONTEXT", "EVIDENCE_REASON"]:
        if hasattr(interval, field):
            output_field = "CANDIDATE_" + field

            if field == "CANDIDATE_ID":
                output_field = field

            base[output_field] = getattr(interval, field)

    base.update(LOT_START=lot.START, LOT_END=lot.END, LOT_ROWS=lot.ROWS,
                STATUS="review", CAUSALITY_CONFIRMED=False)

    context_requests.append({
        "base": base, "records": context_equipment, "kind": "equipment",
        "source_file": "G-01_설비마스터.csv", "source_id": "EQP_TAG", "timing": "",
    })

    categories = sorted(set(["BM", "TBM", "CBM"]) | set(context_maint.MNT_TYPE.dropna()))

    for category in categories:
        records = context_maint.loc[context_maint.MNT_TYPE.eq(category)]

        for kind, time_col, end_col in [("maintenance_work", "STRT_DT", "END_DT"),
                                       ("maintenance_detection", "DETC_DT", None)]:
            timing = category + ": " + time_col

            if end_col:
                timing += "~END_DT [시작,종료)"
            else:
                timing += " 시점"

            context_requests.append({
                "base": base, "records": records, "kind": kind,
                "source_file": "G-02_정비이력.csv | G-02_부품교체이력.csv | G-01_부품마스터.csv",
                "source_id": "WO_NO", "timing": timing, "category": category,
                "time_col": time_col, "end_col": end_col,
            })

    for include_end_date in [False, True]:
        calendar = context_repair.copy()
        extension = pd.Timedelta(0)
        basis = "원본 END_DT 자정까지"

        if include_end_date:
            extension = pd.Timedelta(days=1)
            basis = "종료일 포함 가정; END_DT 다음 날 자정까지"

        calendar["CONTEXT_END"] = calendar.END_DT + extension

        context_requests.append({
            "base": base, "records": calendar, "kind": "repair_calendar",
            "source_file": "G-01_정기수리캘린더.csv", "source_id": "PLAN_ID", "timing": basis,
            "time_col": "STRT_DT", "end_col": "CONTEXT_END",
        })

    event_types = sorted(context_events.EVT_TYPE.dropna().unique())

    if not event_types:
        event_types = [None]

    for event_type in event_types:
        records = context_events.loc[context_events.EVT_TYPE.eq(event_type)]

        context_requests.append({
            "base": base, "records": records, "kind": "operation_event",
            "source_file": "G-02_조업이벤트.csv", "source_id": "EVT_ID",
            "timing": str(event_type) + ": EVT_DT 시점", "event_type": event_type,
            "time_col": "EVT_DT", "end_col": None,
        })


### 직전, 구간 중, 직후 기록 선택

시간 차이와 동률을 남기며, 기록이 없으면 빈 행으로 둠.


In [ ]:
# 작업은 [시작, 종료), 감지와 조업이벤트는 한 시점으로 처리함.
# 직전과 직후의 가장 가까운 기록은 동률도 전부 남김. 겹치는 기록도 전부 남김.
selected_context_records = []

for request in context_requests:
    records = request["records"]
    start = request["base"]["START"]
    end = request["base"]["END"]

    if request["kind"] == "equipment":
        record = None

        if not records.empty:
            record = records.iloc[0]

        selected_context_records.append((request, record, "metadata", None))
        continue

    if records.empty:
        for relation in ["before", "overlap", "after"]:
            selected_context_records.append((request, None, relation, None))

        continue

    starts = pd.to_datetime(records[request["time_col"]], errors="raise")
    ends = starts

    if request["end_col"] is not None:
        ends = pd.to_datetime(records[request["end_col"]], errors="raise")

    point = ends.eq(starts)
    before = ends.lt(start) | (ends.eq(start) & ~point)
    after = starts.gt(end)
    overlap = ~(before | after)

    for relation, mask in [("before", before), ("overlap", overlap), ("after", after)]:
        subset = records.loc[mask]

        if subset.empty:
            selected_context_records.append((request, None, relation, None))
            continue

        if relation == "before":
            subset = subset.loc[ends.loc[subset.index].eq(ends.loc[subset.index].max())]
        elif relation == "after":
            subset = subset.loc[starts.loc[subset.index].eq(starts.loc[subset.index].min())]

        for index, record in subset.iterrows():
            gap = pd.Timedelta(0)

            if relation == "before":
                gap = start - ends.loc[index]
            elif relation == "after":
                gap = starts.loc[index] - end

            selected_context_records.append((request, record, relation, gap.total_seconds() / 3600))


### 원본 기록과 판단 근거


In [ ]:
# 원본 필드는 SOURCE_ 접두어로 남기고 근접 관계와 이유를 별도 컬럼에 기록함.
context_rows = []

for request, record, relation, gap in selected_context_records:
    kind = request["kind"]
    timing = request["timing"]

    if kind == "equipment":
        reason = "온도 CSV 대상 CAL01의 설비 정보임. 센서별 위치·단위·설정값 마스터는 원본에 없음"
    elif kind in ["maintenance_work", "maintenance_detection"]:
        category = request["category"]
        reason = category + " " + relation + " 기록 없음"

        if record is not None:
            part_names = record.get("PART_NAMES", "") or "교체 기록 없음"
            reason = (
                category + " " + relation + ": " + str(record.WO_NO) + "; 감지 " + str(record.DETC_DT) + "; "
                + "작업 " + str(record.STRT_DT) + "~" + str(record.END_DT) + "; 고장코드 " + str(record.FAIL_MODE) + "; "
                + "기록 " + str(record.REMARK) + "; 부품 " + str(part_names) + "; "
                + "교체사유 " + str(record.get("PART_CHANGE_REASONS", "")) + "; DOWN_MIN=" + str(record.DOWN_MIN) + ". "
                + "시간상 근접·겹침이며 센서 변화의 원인 확정 아님. 정지 시점은 역산하지 않음"
            )
    elif kind == "repair_calendar":
        reason = "해당 방향의 CR1 정기수리 기록 없음"

        if record is not None:
            reason = (str(record.PLAN_ID) + ": " + str(record.SCOPE) + "; " + timing
                      + ". 예정 범위이며 실제 정지·작업 이행 여부는 이 표만으로 확정 불가")
    else:
        reason = "CR1 " + str(request["event_type"]) + " " + relation + " 기록 없음"

        if record is not None:
            reason = (str(record.EVT_ID) + ": " + str(record.EVT_TYPE) + " "
                      + str(record.BEF_VAL) + "→" + str(record.AFT_VAL)
                      + "; 운전 조건 맥락이며 센서 이상 원인 확정 아님")

    source_id = None

    if record is not None:
        source_id = record[request["source_id"]]

    output = dict(request["base"], SOURCE_KIND=kind, SOURCE_FILE=request["source_file"],
                  SOURCE_ID=source_id, RELATION=relation, GAP_HOURS=gap, TIMING_BASIS=timing,
                  EVIDENCE_FOUND=record is not None, EVIDENCE_REASON=reason)

    if record is not None:
        for field, value in record.items():
            output["SOURCE_" + field] = value

    context_rows.append(output)

domain_context = pd.DataFrame(context_rows)

if domain_context.empty:
    domain_context = pd.DataFrame(columns=[
        "CONTEXT_ROW_ID", "INTERVAL_ID", "LOT_NO", "GRADE", "SENSOR", "START", "END", "SOURCE_KIND",
        "SOURCE_ID", "RELATION", "GAP_HOURS", "EVIDENCE_FOUND", "EVIDENCE_REASON", "STATUS",
    ])
else:

    context_ids = []

    for number in range(1, len(domain_context) + 1):
        context_ids.append("DCTX-" + str(number).zfill(5))

    domain_context.insert(0, "CONTEXT_ROW_ID", context_ids)


### 구간 ID로 근거 연결


In [ ]:
domain_context['MODEL_INPUT_ALLOWED'] = False

# 기존 LOT 검토표에서도 같은 정비 근거를 사용함.
lot_evidence = domain_context.loc[
    domain_context.CANDIDATE_KIND.eq('LOT_CONTEXT_ONLY')
    & domain_context.SOURCE_KIND.isin(['maintenance_work', 'maintenance_detection'])
    & domain_context.EVIDENCE_FOUND
].copy()

lot_evidence['CONTEXT_KIND'] = lot_evidence.SOURCE_KIND + ':' + lot_evidence.RELATION

lot_evidence = lot_evidence.sort_values(['LOT_NO', 'GAP_HOURS']).drop_duplicates(['LOT_NO', 'SOURCE_WO_NO'])

maintenance_columns = [
    'WO_NO', 'MNT_TYPE', 'DETC_DT', 'STRT_DT', 'END_DT', 'FAIL_MODE', 'REMARK',
    'DOWN_MIN', 'WORK_MIN', 'DOWN_RECORD_STATUS', 'DOWN_TO_WORK_RATIO',
    'DOWN_EXCEEDS_WORK', 'STOP_INTERVAL_KNOWN', 'PART_CHG', 'HAS_PART_RECORD',
    'PART_RECORD_COUNT', 'PART_CODES', 'PART_NAMES', 'PART_QTY_TOTAL', 'PART_CHANGE_REASONS',
]

source_columns = []
rename_columns = {}

for column in maintenance_columns:
    source_column = 'SOURCE_' + column
    source_columns.append(source_column)
    rename_columns[source_column] = column

lot_maintenance_evidence = lot_evidence[
    ['LOT_NO', 'GRADE', 'CONTEXT_KIND', 'GAP_HOURS'] + source_columns
].rename(columns=rename_columns).sort_values(['LOT_NO', 'DETC_DT'])

lot_maintenance_evidence['EVIDENCE_TYPE'] = '정비·부품교체 사후 맥락'
lot_maintenance_evidence['REVIEW_STATUS'] = '미확정'
lot_maintenance_evidence['MODEL_INPUT_ALLOWED'] = False

# 하나의 행에서 해당 LOT 맥락과 센서 후보의 근거를 찾아갈 수 있게 연결함.
domain_row_index = retained[['LOT_NO', 'MEAS_DT', 'GRADE']].sort_values(['LOT_NO', 'MEAS_DT']).reset_index(drop=True)
lot_ids = domain_intervals.loc[domain_intervals.KIND.eq('LOT_CONTEXT_ONLY')].set_index('LOT_NO').INTERVAL_ID

domain_row_index['LOT_CONTEXT_ID'] = domain_row_index.LOT_NO.map(lot_ids)
domain_row_index['SENSOR_CANDIDATE_IDS'] = ''

for interval in domain_intervals.loc[~domain_intervals.KIND.eq('LOT_CONTEXT_ONLY')].itertuples(index=False):
    hit = domain_row_index.LOT_NO.eq(interval.LOT_NO) & domain_row_index.MEAS_DT.between(interval.START, interval.END)
    domain_row_index.loc[hit, 'SENSOR_CANDIDATE_IDS'] += interval.INTERVAL_ID + '|'

domain_row_index['SENSOR_CANDIDATE_IDS'] = domain_row_index.SENSOR_CANDIDATE_IDS.str.rstrip('|')

domain_row_index['LABEL'] = pd.Series(pd.NA, index=domain_row_index.index, dtype='Int8')
domain_row_index['REVIEW_STATUS'] = '미판정: 센서·운전조건·CSV 사건 근거 종합 필요'

display(lot_maintenance_evidence)
print('검토 구간:', len(domain_intervals), '/ 원본 행 근거 연결:', len(domain_row_index))

## 6. 정상, 이상 판정은 미확정

`REASON`과 `EVIDENCE_TYPE`은 검토 근거임. `MODEL_INPUT_ALLOWED=False`는 근거표를 그대로 모델 입력에 쓰지 않는다는 뜻임.

`LABEL`은 `NA`, `REVIEW_STATUS`는 미확정으로 남김. 이상 후보가 없다는 이유만으로 정상 0을 채우지 않음.


In [ ]:
row_review["CP_CANDIDATE_CALCULATED"] = CALCULATE_CP_CANDIDATES
row_review["LABEL"] = pd.Series(pd.NA, index=row_review.index, dtype="Int8")
row_review["REVIEW_STATUS"] = "미확정"
row_review["EVIDENCE_TYPE"] = "판정 대기"
row_review["LABEL_REASON"] = "도메인 기준과 구간 판정 미확정. 정상 0으로 확정한 행이 아님"

row_review.loc[row_review["CP_CANDIDATE"].fillna(False), ["EVIDENCE_TYPE", "LABEL_REASON"]] = [
    "CP 신호 후보", "CP 지속 이탈 후보임. 공정 허용치와 정비 맥락 확인 전 이상 1로 확정하지 않음"
]

row_review.loc[row_review["EXCLUDED"], ["EVIDENCE_TYPE", "LABEL_REASON"]] = [
    "사용자 지정 시간 제외", "사용자가 지정한 시간 범위여서 제외함. 정상·이상 판정과 별개임"
]

review_notes = {
    240001: "TC·OP 반복 변동만으로 이상 확정하지 않음. 원인 확인 필요함",
    240024: "OP 하강·TC 상승·CP 변동 관계를 같이 확인해야 함",
    240037: "OP 반복 변동만으로 이상 확정하지 않음. CP 후보는 따로 검토함",
    240038: "TC·OP 반응과 CP 경계의 작은 초과를 구분해서 확인해야 함",
    240044: "초반·후반 OP 결측 처리와 후속 BM 내용을 따로 확인해야 함",
    240058: "08:00 이후 제외 확정. 앞쪽 CP 저하가 정기수리 준비 과정인지 확인해야 함",
    240061: "02:30 이전 제외 확정. 남은 출력만으로 이상 확정하지 않음",
    240069: "CP 변화와 절연 저하·O링 교체를 같이 보되 가스 누출로 확정하지 않음",
    240091: "같은 SPHC보다 TC-Z7/Z8이 높게 지속됨. 짧은 급변과 별도로 설정·운전조건 및 절연 저하 기록을 검토해야 함",
    240108: "OP 변동만으로 원인 확정하지 않음. 정비 기록과 같이 검토함",
    240114: "20:00 이후 제외 확정. 이전 TC·OP 제어 반응을 검토함",
    240117: "09:00 이전 제외 확정. 정비 후 CP 저하의 작업 맥락 확인이 필요함",
    240121: "급격한 TC 변화가 운전 반응인지 이상인지 기준 확인이 필요함",
    240133: "CBM 감지 사실과 실제 센서 구간을 나눠서 검토해야 함",
}

lot_review_rows = []

for lot, group in row_review.groupby("LOT_NO", sort=True):
    lot_review_rows.append({
        "LOT_NO": lot, "TOTAL_ROWS": len(group), "EXCLUDED_ROWS": group["EXCLUDED"].sum(),
        "CP_CANDIDATE_ROWS": int(group["CP_CANDIDATE"].sum()),
        "CP_NOT_EVALUATED_ROWS": int(group["CP_CANDIDATE"].isna().sum()),
        "CONFIRMED_LABEL_ROWS": int(group["LABEL"].notna().sum()),
    })

lot_review = pd.DataFrame(lot_review_rows)
# 기존 집계 결과의 nullable 정수 형식을 유지함.
lot_review["CP_NOT_EVALUATED_ROWS"] = lot_review["CP_NOT_EVALUATED_ROWS"].astype("Int64")
lot_review["CONFIRMED_LABEL_ROWS"] = lot_review["CONFIRMED_LABEL_ROWS"].astype("Int8")

lot_review["RETAINED_ROWS"] = lot_review["TOTAL_ROWS"] - lot_review["EXCLUDED_ROWS"]

lot_review = lot_context.merge(lot_review, on="LOT_NO", how="left", validate="one_to_one")

related_wo = {}

for lot, group in lot_maintenance_evidence.groupby("LOT_NO", sort=True):
    related_wo[lot] = " | ".join(group["WO_NO"])

lot_review["CONTEXT_WO_NOS"] = lot_review["LOT_NO"].map(related_wo).fillna("")
lot_review["CP_CANDIDATE_CALCULATED"] = CALCULATE_CP_CANDIDATES

if not CALCULATE_CP_CANDIDATES:
    lot_review["CP_CANDIDATE_ROWS"] = pd.Series(pd.NA, index=lot_review.index, dtype="Int64")

lot_review["REASON"] = lot_review["LOT_NO"].map(review_notes).fillna("별도 도메인 검토 필요함")
lot_review["EVIDENCE_TYPE"] = "LOT·강종·정비 맥락 검토"
lot_review["REVIEW_STATUS"] = "미확정"
lot_review["MODEL_INPUT_ALLOWED"] = False

display(lot_review)

## 7. 검토 결과 저장

원본 센서를 복제하지 않고 `LOT_NO`, `MEAS_DT`로 연결할 행별 검토, 제외 구간, 이상 후보와 정비 근거를 CSV로 저장함.

확정 판정이 생기면 `review/v1/05_confirmed_intervals.csv`에 `GRADE`, `LOT_NO`, `START`, `END`, `LABEL`, `REASON`, `EVIDENCE`, `REVIEWER`를 기록함. 현재는 확정표를 임의로 만들거나 `NA`를 정상 0으로 바꾸지 않음.


In [ ]:
REVIEW_COLUMNS = [
    "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "EXCLUSION_REASON",
    "CP_CANDIDATE_CALCULATED", "CP_CANDIDATE", "CP_RUN_ID", "LABEL", "REVIEW_STATUS", "EVIDENCE_TYPE", "LABEL_REASON",
]

review_export = row_review[REVIEW_COLUMNS].copy()

outputs = {
    "05_row_review.csv": review_export,
    "05_domain_intervals.csv": domain_intervals,
    "05_domain_context.csv": domain_context,
    "05_domain_row_index.csv": domain_row_index,
    "05_exclusion_summary.csv": exclusion_summary,
    "05_all_sensor_sustained_candidates.csv": sustained_candidates,
    "05_temporal_channel_coverage.csv": temporal_coverage,
    "05_lot_review.csv": lot_review,
    "05_lot_maintenance_evidence.csv": lot_maintenance_evidence,
    "05_grade_sensor_review.csv": grade_sensor_review,
    "05_lot_sensor_summary.csv": lot_sensor_summary,
}

output_directories = {
    "05_grade_sensor_review.csv": GRADE_OUT,
    "05_lot_sensor_summary.csv": GRADE_OUT,
    "05_all_sensor_sustained_candidates.csv": SIGNAL_OUT,
    "05_temporal_channel_coverage.csv": SIGNAL_OUT,
}

for filename, frame in outputs.items():
    directory = output_directories.get(filename, OUT)
    output_path = os.path.join(directory, filename)
    frame.to_csv(output_path, index=False, encoding=ENC)

output_summary = []

for name, frame in outputs.items():
    output_summary.append({
        "파일": name,
        "행수": len(frame),
        "저장 위치": output_directories.get(name, OUT),
    })

display(pd.DataFrame(output_summary))
print("확정 라벨 행수:", int(row_review["LABEL"].notna().sum()))

import json

rules_path = os.path.join(SIGNAL_OUT, '05_temporal_review_rules.json')

with open(rules_path, 'w', encoding='utf-8') as file:
    json.dump(TEMPORAL_REVIEW_RULES, file, ensure_ascii=False, indent=2)